In [ ]:
!pip install -q -U transformers==4.38.2
!pip install -q -U datasets==2.18.0
!pip install -q -U accelerate==0.27.2
!pip install -q -U peft==0.9.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6

In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

# ==========================================
# 1. SETUP DATA (From original notebook)
# ==========================================
data_path = "https://raw.githubusercontent.com/bitext/customer-support-llm-chatbot-training-dataset/main/data/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
df = pd.read_csv(data_path)
df = df.head(10000) # Keep small for the lab demo Code.ipynb]

# Format Data into Instruction/Response pairs Code.ipynb]
training_data = []
for i in range(len(df)):
    entry = {'text': f"### Question:\n{df.iloc[i]['instruction']}\n\n### Answer:\n{df.iloc[i]['response']}"}
    training_data.append(entry)

dataset = Dataset.from_pandas(pd.DataFrame(training_data))

In [13]:
df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [ ]:
# ==========================================
# 2. LOAD MODEL & TOKENIZER (FP16)
# ==========================================
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load base model in Float16 to save 50% memory
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Tokenize the dataset Code.ipynb]
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.add_column("labels", tokenized_datasets["input_ids"])
split_dataset = tokenized_datasets.train_test_split(test_size=0.1, seed=42)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
q_proj_count = 0
v_proj_count = 0

for name, module in base_model.named_modules():
    if "q_proj" in name:
        q_proj_count += 1
    if "v_proj" in name:
        v_proj_count += 1

In [ ]:
print(f"Number of 'q_proj' layers: {q_proj_count}")
print(f"Number of 'v_proj' layers: {v_proj_count}")
print(f"Total targeted layers for LoRA: {q_proj_count + v_proj_count}")

Number of 'q_proj' layers: 22
Number of 'v_proj' layers: 22
Total targeted layers for LoRA: 44


In [ ]:
# Attention projections (q_proj, k_proj, v_proj, o_proj)
# q : what to focus on
# k : matching
# v : information flow
# o : output mixing

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r = 8,
    lora_alpha = 16,
    lora_dropout = 0.05,
    target_modules = ["q_proj","v_proj"]
)

In [ ]:
# wrap the base model with config
peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# Train :
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs = 1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4, # simulates batch size of 4
    learning_rate = 2e-5,
    max_grad_norm = 0.3,
    logging_steps= 10,
    optim = "adamw_torch",
    save_strategy = "no")

trainer = Trainer(model = peft_model,
                  args = training_args,
                  train_dataset = split_dataset['train'],
                  eval_dataset  = split_dataset['test'])

trainer.train()

Step,Training Loss
10,8.032822
20,7.714143
30,7.217583
40,6.352675
50,5.760689
60,4.419481
70,4.377611
80,3.065674
90,2.984521
100,2.070190


TrainOutput(global_step=2250, training_loss=0.8856016529930962, metrics={'train_runtime': 1075.6378, 'train_samples_per_second': 8.367, 'train_steps_per_second': 2.092, 'total_flos': 1.4316670550016e+16, 'train_loss': 0.8856016529930962, 'epoch': 1.0})

In [ ]:
def run_inference(question, model):
    prompt = f"### Question:\n{question}\n\n### Answer:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate output
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n--- Output ---")
print(run_inference("I want to update my billing address.", peft_model))


--- Output ---
### Question:
I want to update my billing address.

### Answer:
I'm here to help you update your billing address. To make the process as smooth as possible, please provide me with your billing address details.


In [ ]:
print(run_inference("I want to cancel my order?", peft_model))

### Question:
I want to cancel my order?

### Answer:
I'm sorry to hear that you're looking to cancel your order. I'm here to assist you with that. To cancel your order, please follow these steps:

1. Log in to your account on our website.
2. Navigate to the "Order History" section.
3. Locate the order that you want to cancel.
4. Click on the "Cancel Order" button.
5. A confirmation message will appear, indicating that


In [ ]:
print(run_inference("My payment failed, what should I do?", peft_model))

### Question:
My payment failed, what should I do?

### Answer:
I'm sorry to hear that your payment failed. We understand that this can be frustrating, and we're here to help you resolve the issue. To assist you further, could you please provide me with more information about the payment you attempted? This will help us identify the specific error and provide you with the necessary guidance.


In [ ]:
print(run_inference("How do I cancel my subscription?", peft_model))

### Question:
How do I cancel my subscription?

### Answer:
I'm sorry to hear that you're facing difficulties with canceling your subscription. To cancel your subscription, please follow these steps:

1. Log in to your account on our website.
2. Navigate to the "Subscription" or "Membership" section.
3. Look for the subscription you want to cancel.
4. Click on the subscription you want to cancel.
5. A pop-up will appear asking you to confirm the can
